# Torch performance investigation
This notebook tries to query torch into giving more information about the compilation of (CPU) kernels via `torch.compile()`. I need to understand why my C++ implementations are consistently faster than the pytorch ones.

Here are three possible strategies for validating the claim:
  1. inspect the torch inductor pipeline (and possibly the generated sources)
  2. compile the C++ implementations at runtime using the pytorch compilation pipeline
  3. profile the hardware and test for missing vectorization or redundant memory allocations

## Torch inductor exploration

In [1]:
import torch

In [2]:
torch._dynamo.list_backends()

['cudagraphs', 'inductor', 'openxla', 'tvm']

In [3]:
torch._inductor.list_mode_options()

{'default': {},
 'reduce-overhead': {'triton.cudagraphs': True},
 'max-autotune-no-cudagraphs': {'max_autotune': True,
  'coordinate_descent_tuning': True},
 'max-autotune': {'max_autotune': True,
  'triton.cudagraphs': True,
  'coordinate_descent_tuning': True}}

In [4]:
torch._inductor.list_options()

['TYPE_CHECKING',
 'inplace_padding',
 'can_inplace_pad_graph_input',
 'enable_auto_functionalized_v2',
 'debug',
 'disable_progress',
 'verbose_progress',
 'worker_log_path',
 'precompilation_timeout_seconds',
 'fx_graph_cache',
 'remote_gemm_autotune_cache',
 'fx_graph_remote_cache',
 'bundle_triton_into_fx_graph_cache',
 'non_blocking_remote_cache_write',
 'autotune_local_cache',
 'autotune_remote_cache',
 'bundled_autotune_remote_cache',
 'unsafe_skip_cache_dynamic_shape_guards',
 'unsafe_marked_cacheable_functions',
 'sleep_sec_TESTING_ONLY',
 'triton_kernel_default_layout_constraint',
 'cpp_wrapper',
 'cpp_wrapper_build_separate',
 'fx_wrapper',
 'cpp_cache_precompile_headers',
 'online_softmax',
 'dce',
 'static_weight_shapes',
 'size_asserts',
 'nan_asserts',
 'scalar_asserts',
 'alignment_asserts',
 'pick_loop_orders',
 'inplace_buffers',
 'allow_buffer_reuse',
 'memory_planning',
 'use_fast_math',
 'bfloat16_atomic_adds_enabled',
 'memory_pool',
 'benchmark_harness',
 'epilog

## Kernel code

In [5]:
import numpy as np
from slmporting.core.slm import SLM
from slmporting.core.types import Backend

In [6]:
n = 100
x = torch.rand(n)
y = torch.rand(n)
z = torch.rand(n)
pists = torch.rand(n)
slm = SLM.get_standard_slm()

/home/matteo/github/slm-porting/venv/lib/python3.12/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [7]:
ε = None

# here I replicate a pytorch kernel for ease of access
def rs(x, y, z, pists, xx, yy, C1: float, C2: float):
    return torch.angle(
        torch.mean(
            torch.exp(
                1j * (
                    C1 * (x[:,ε]*xx[ε,:] + y[:,ε]*yy[ε,:]) +
                    C2 * z[:,ε] * (xx**2 + yy**2)[ε,:] +
                    2*np.pi*pists[:,ε]
                )
            ),
            dim=0
        )
    )


def rs_no_complex(x, y, z, pists, xx, yy, C1: float, C2: float):
    slm_p_phase = C1 * (x[:,ε]*xx[ε,:] + y[:,ε]*yy[ε,:]) + \
                  C2 * z[:,ε] * (xx**2 + yy**2)[ε,:] + \
                  2*np.pi*pists[:,ε]

    avg_field = torch.vstack((
        torch.mean(torch.cos(slm_p_phase), dim=0),
        torch.mean(torch.sin(slm_p_phase), dim=0)
    ))

    return torch.arctan2(avg_field[1], avg_field[0])

## Loading C++ sources from torch
ChatGPT suggests to "monkey patch" the `AsyncCompile` function. This should print the code before compiling

In [8]:
from torch._inductor.async_compile import AsyncCompile
orig_cpp = AsyncCompile.cpp_pybinding

def capture_cpp(self, argtypes: list[str], source_code: str):
    print(source_code)
    return orig_cpp(self, argtypes, source_code)

AsyncCompile.cpp_pybinding = capture_cpp

In [9]:
torch.compile(rs)(x, y, z, pists, slm.xx.convert_to(Backend.TORCH), slm.yy.convert_to(Backend.TORCH), slm.C1, slm.C2)

/home/matteo/github/slm-porting/venv/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
W1101 14:07:09.729000 16997 torch/utils/cpp_extension.py:117] [0/0] No CUDA runtime is found, using CUDA_HOME='/usr'



#include <torch/csrc/inductor/cpp_prefix.h>
extern "C"  void  kernel(const float* in_ptr0,
                       const double* in_ptr1,
                       const float* in_ptr2,
                       const double* in_ptr3,
                       const float* in_ptr4,
                       const float* in_ptr5,
                       double* out_ptr0)
{
    #pragma omp parallel num_threads(10)
    {
        int tid = omp_get_thread_num();
        {
            #pragma omp for
            for(int64_t x0=static_cast<int64_t>(0L); x0<static_cast<int64_t>(100L); x0+=static_cast<int64_t>(1L))
            {
                for(int64_t x1=static_cast<int64_t>(0L); x1<static_cast<int64_t>(205012L); x1+=static_cast<int64_t>(8L))
                {
                    {
                        if(C10_LIKELY(x1 >= static_cast<int64_t>(0) && x1 < static_cast<int64_t>(205008L)))
                        {
                            auto tmp0 = in_ptr0[static_cast<int64_t>(x0)];
               

tensor([-2.1427, -2.1380, -2.1333,  ..., -2.7812, -2.7694, -2.7576],
       dtype=torch.float64)

In [10]:
torch.compile(rs_no_complex)(x, y, z, pists, slm.xx.convert_to(Backend.TORCH), slm.yy.convert_to(Backend.TORCH), slm.C1, slm.C2)


#include <torch/csrc/inductor/cpp_prefix.h>
extern "C"  void  kernel(const float* in_ptr0,
                       const double* in_ptr1,
                       const float* in_ptr2,
                       const double* in_ptr3,
                       const float* in_ptr4,
                       const float* in_ptr5,
                       const double* in_ptr6,
                       double* out_ptr0,
                       double* out_ptr1,
                       double* out_ptr2,
                       double* out_ptr3,
                       double* out_ptr4,
                       double* out_ptr5)
{
    #pragma omp parallel num_threads(10)
    {
        int tid = omp_get_thread_num();
        {
            #pragma omp for
            for(int64_t x0=static_cast<int64_t>(0L); x0<static_cast<int64_t>(100L); x0+=static_cast<int64_t>(1L))
            {
                for(int64_t x1=static_cast<int64_t>(0L); x1<static_cast<int64_t>(205012L); x1+=static_cast<int64_t>(8L))
             

tensor([-2.1427, -2.1380, -2.1333,  ..., -2.7812, -2.7694, -2.7576],
       dtype=torch.float64)

In [ ]:
def rs_ssa(x, y, z, pists, xx, yy, C1: float, C2: float):
    slm_p_phase = C1 * (x[:,ε]*xx[ε,:] + y[:,ε]*yy[ε,:]) + \
                  C2 * z[:,ε] * (xx**2 + yy**2)[ε,:] + \
                  2*np.pi*pists[:,ε]

    # if I use the `slm_p_phase` tensor a single time
    # the compiler can perform the reduction in place
    sines   = torch.sum(torch.sin(slm_p_phase), dim=0)
    cosines = torch.sum(torch.cos(slm_p_phase), dim=0)

    return torch.arctan2(sines, cosines)

torch.compile(rs_ssa)(x, y, z, pists, slm.xx.convert_to(Backend.TORCH), slm.yy.convert_to(Backend.TORCH), slm.C1, slm.C2)

In [19]:
def rs_for_loop(x, y, z, pists, xx, yy, C1: float, C2: float):
    sines = torch.zeros_like(xx)
    cosines = torch.zeros_like(xx)

    for i in range(x.shape[0]):
        slm_p_phase = C1 * (x[i]*xx + y[i]*yy) + \
                      C2 * z[i] * (xx**2 + yy**2) + \
                      2*np.pi*pists[i]

        sines += torch.sin(slm_p_phase)
        cosines += torch.cos(slm_p_phase)

    return torch.arctan2(sines, cosines)

torch.compile(rs_for_loop)(x, y, z, pists, slm.xx.convert_to(Backend.TORCH), slm.yy.convert_to(Backend.TORCH), slm.C1, slm.C2)


#include <torch/csrc/inductor/cpp_prefix.h>
extern "C"  void  kernel(double* in_out_ptr0,
                       double* in_out_ptr1,
                       const float* in_ptr0,
                       const double* in_ptr1,
                       const float* in_ptr2,
                       const double* in_ptr3,
                       const float* in_ptr4,
                       const float* in_ptr5,
                       double* out_ptr1,
                       double* out_ptr2,
                       double* out_ptr3,
                       double* out_ptr4,
                       double* out_ptr5,
                       double* out_ptr6,
                       double* out_ptr7,
                       double* out_ptr8,
                       double* out_ptr10,
                       double* out_ptr11,
                       double* out_ptr12,
                       double* out_ptr13,
                       double* out_ptr14,
                       double* out_ptr15,
             

tensor([-2.1427, -2.1380, -2.1333,  ..., -2.7812, -2.7694, -2.7576],
       dtype=torch.float64)